# Water Quality Surrogate Model

This example demonstrates how to use the water quality surrogate model proposed in *MeGA-MP: Metric Graph Advection Message Passing*, Janine Strotherm, Luca Hermes, André Artelt, Barbara Hammer, Transactions of Machine Learning Research (2026), https://openreview.net/forum?id=2aZPFrKYYb.
Note that this surrogate model supports single species scenarios only.

In this example, we show how to setup and use the surrogate model predicting chlorine (CL2) concentrations at every node in the Hanoi network based on a setpoint source at the reservoir.

In [ ]:
from IPython.display import display, HTML
display(HTML('<a target=\"_blank\" href=\"https://colab.research.google.com/github/WaterFutures/EPyT-Control/blob/main/docs/examples/quality_surrogate.ipynb\"><img src=\"https://colab.research.google.com/assets/colab-badge.svg\" alt=\"Open In Colab\"/></a>'))

Because of the large depenecies, the surrogate model is not included by default but must be explicitly installed by either installing all optional dependencies (`[all]`) or only this surrogate model (`[qualsurrogate]`):

In [ ]:
%pip install epyt-control[qualsurrogate] --quiet

In [ ]:
import numpy as np
from epyt_control.models import QualitySurrogateModel
from epyt_flow.utils import to_seconds, plot_timeseries_prediction, plot_timeseries_data
from epyt_flow.data.networks import load_hanoi

from sim_utils import run_simulation

This example is based on Hanoi.

In [ ]:
f_inp_in = load_hanoi().f_inp_in
f_msx_in = "simplecl2.msx"  # No topological information included! Can be used for any network!
sources_at = ['1']   # Chlorine source at the reservoir

In [ ]:
N_SECONDS = to_seconds(hours=12)
HYDRAULIC_STEP = 60
QUALITY_TIMESTEP = 1
nsteps = int(N_SECONDS / HYDRAULIC_STEP)
times = np.linspace(0, N_SECONDS, nsteps)

Create a random wave pattern to model the concentration of CL2 at the source node.

In [ ]:
def generate_injection_pattern(num_points, min_modes=3, max_modes=30, seed=None):
    rs = np.random.RandomState(seed)
    
    t = np.linspace(0, 2 * np.pi, num_points)
    signal = np.zeros(num_points)
    num_modes = rs.randint(min_modes, max_modes)
    decay_power = rs.uniform(0.5, 1.5)
    
    for k in range(1, num_modes + 1):
        amplitude_scale = 1.0 / (k**decay_power)
        a_k = rs.normal() * amplitude_scale
        b_k = rs.normal() * amplitude_scale
        signal += a_k * np.cos(k * t) + b_k * np.sin(k * t)
        
    return (signal - signal.min()) / (signal.max() - signal.min())

In [ ]:
injection_pattern = generate_injection_pattern((N_SECONDS // HYDRAULIC_STEP) + 1, seed=42)

plot_timeseries_data(injection_pattern.reshape(1, -1), x_axis_label="Time steps", y_axis_label="CL2 concentration at source node")

Run hydraulic simulation to get the flow rates and ground truth concentrations at every node

In [ ]:
sim_kwargs = {
    'duration' : N_SECONDS,
    'hydraulic_step' : HYDRAULIC_STEP,
    'quality_step' : QUALITY_TIMESTEP,
    'f_msx_in' : f_msx_in,
    'source_conc' : {n_id: injection_pattern for n_id in sources_at}
}

scada_data = run_simulation(f_inp_in, **sim_kwargs)  # Uses EPyT-Flow for running EPANET and EPANET-MSX simulations

# Extrac ground truth concentration
if sim_kwargs["f_msx_in"] is None:
    conc = scada_data.get_data_nodes_quality()
else:
    conc = scada_data.get_data_bulk_species_node_concentration()

#### Build water quality surrogate model

For this, we simply create a new instance of [epyt_control.models.QualitySurrogateModel](https://epyt-control.readthedocs.io/en/stable/epyt_control.models.html#epyt_control.models.qualsurrogate_mega_mp.quality_surrogate.QualitySurrogateModel) and pass the [network topology](https://epyt-flow.readthedocs.io/en/stable/epyt_flow.html#epyt_flow.topology.NetworkTopology) (of the network for which we want to build the surrogate) as an argument to the constructur:

In [ ]:
surrogate = QualitySurrogateModel(scada_data.network_topo)

We can now use the surrogate to predict the concentration of our single species (here, Chlorine) at all nodes, based on one or multiple sources -- the concentration at those sources is set to a given pattern.

The [QualitySurrogateModel.predict()](https://epyt-control.readthedocs.io/en/stable/epyt_control.models.html#epyt_control.models.qualsurrogate_mega_mp.quality_surrogate.QualitySurrogateModel.predict) function requires two arguments:
1. A dictionary of the concentration (value) at each source (key).
2. The [SCADA data](https://epyt-flow.readthedocs.io/en/stable/epyt_flow.simulation.scada.html#epyt_flow.simulation.scada.scada_data.ScadaData) of the hydraulic simulation, containing the flow rates at all links.

In [ ]:
conc_pred = surrogate.predict({"1": injection_pattern}, scada_data)  # Specify source concentration at node "1"
print(conc_pred.shape)  # 1. dimension: time; 2. dimension: nodes

#### Evaluation

In [ ]:
print(f"MAE: {np.mean(np.abs(conc_pred - conc))}")

In [ ]:
# Plot ground truth concentration vs. prediction from the surrogate model at six random nodes
for idx in np.random.RandomState(32).choice(range(scada_data.network_topo.get_number_of_nodes()), size=6):
    plot_timeseries_prediction(conc[:, idx], conc_pred[:, idx],
                               x_axis_label="Time steps", y_axis_label="CL2 concentration")